# Exploração do IBGE - População Municipal

Este notebook realiza a exploração inicial dos dados de população municipal disponibilizados pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

Nesta etapa serão analisados:

- arquivo de estimativa populacional de 2021;
- estrutura das colunas;
- códigos dos municípios;
- nomes dos municípios;
- população estimada;
- municípios do estado de Goiás;
- compatibilidade dos códigos municipais com os dados do SIH/SUS e CNES.

Os dados populacionais serão utilizados posteriormente para construir indicadores relativos à população como internações e disponibilidade de leitos por habitante.

## 1. Importação das bibliotecas

In [1]:
from pathlib import Path
import pandas as pd

## 2. Configuração da análise

Nesta etapa definimos o ano e o arquivo do IBGE utilizado na exploração.

In [2]:
ANO = 2021

arquivo_ibge = Path("../data/raw/ibge/estimativas/2021/estimativa_dou_2021.xls")

print(f"Ano analisado: {ANO}")
print(f"Arquivo: {arquivo_ibge.name}")
print(f"Arquivo encontrado: {arquivo_ibge.exists()}")

Ano analisado: 2021
Arquivo: estimativa_dou_2021.xls
Arquivo encontrado: True


## 3. Planilhas disponíveis

Antes da leitura dos dados, verificamos quais planilhas existem dentro do arquivo Excel.

In [3]:
excel_ibge = pd.ExcelFile(arquivo_ibge)
print("Planilhas encontradas:")

for planilha in excel_ibge.sheet_names:
    print(planilha)

Planilhas encontradas:
BRASIL E UFs
Municípios


## 4. Visualização inicial do arquivo

O arquivo será lido inicialmente sem definir um cabeçalho. Isso permite identificar onde começam os nomes das colunas e onde estão os dados municipais.

In [4]:
df_bruto = pd.read_excel( arquivo_ibge, sheet_name="Municípios", header=None)
df_bruto.head(20)

,0,1,2,3,4
0,ESTIMATIVAS DA POPULAÇÃO RESIDENTE NOS MUNICÍP...,NaN,NaN,NaN,NaN
1,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
2,RO,11,00015,Alta Floresta D'Oeste,22516
3,RO,11,00023,Ariquemes,111148
4,RO,11,00031,Cabixi,5067
5,RO,11,00049,Cacoal,86416
6,RO,11,00056,Cerejeiras,16088
7,RO,11,00064,Colorado do Oeste,15213
8,RO,11,00072,Corumbiara,7052
9,RO,11,00080,Costa Marques,19255


## 5. Identificação do cabeçalho

Nesta etapa procuramos automaticamente a linha que contém os nomes das colunas da tabela.

In [5]:
linha_cabecalho = None

for indice, linha in df_bruto.head(30).iterrows():

    valores = (
        linha
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .tolist()
    )

    if (
        "UF" in valores
        and "COD. UF" in valores
        and "COD. MUNIC" in valores
        and "NOME DO MUNICÍPIO" in valores
        and "POPULAÇÃO ESTIMADA" in valores
    ):
        linha_cabecalho = indice
        break

if linha_cabecalho is None:
    raise ValueError(
        "Não foi possível identificar o cabeçalho da planilha Municípios."
    )

print("Linha identificada como cabeçalho:", linha_cabecalho)

Linha identificada como cabeçalho: 1


## 6. Leitura da tabela

Após identificar o cabeçalho, o arquivo será carregado novamente utilizando os nomes corretos das colunas.

In [6]:
if linha_cabecalho is None:
    raise ValueError(
        "Não foi possível identificar o cabeçalho da planilha Municípios."
    )

df_ibge = pd.read_excel(
    arquivo_ibge,
    sheet_name="Municípios",
    header=linha_cabecalho,
    dtype=str
)

df_ibge.head(10)

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
0,RO,11,00015,Alta Floresta D'Oeste,22516
1,RO,11,00023,Ariquemes,111148
2,RO,11,00031,Cabixi,5067
3,RO,11,00049,Cacoal,86416
4,RO,11,00056,Cerejeiras,16088
5,RO,11,00064,Colorado do Oeste,15213
6,RO,11,00072,Corumbiara,7052
7,RO,11,00080,Costa Marques,19255
8,RO,11,00098,Espigão D'Oeste,33009
9,RO,11,00106,Guajará-Mirim,46930


## 7. Dimensão do dataset

In [7]:
linhas, colunas = df_ibge.shape
print(f"Registros: {linhas:,}")
print(f"Colunas: {colunas}")

Registros: 5,593
Colunas: 5


## 8. Colunas disponíveis

Antes do tratamento dos dados, verificamos os nomes reais das variáveis presentes na tabela do IBGE.

In [8]:
for coluna in df_ibge.columns:
    print(coluna)

UF
COD. UF
COD. MUNIC
NOME DO MUNICÍPIO
POPULAÇÃO ESTIMADA


## 9. Tipos dos dados

In [9]:
df_ibge.dtypes

UF                    object
COD. UF               object
COD. MUNIC            object
NOME DO MUNICÍPIO     object
POPULAÇÃO ESTIMADA    object
dtype: object

## 10. Padronização das colunas

In [10]:
df_ibge = df_ibge.rename(
    columns={
        "UF": "uf",
        "COD. UF": "cod_uf",
        "COD. MUNIC": "cod_municipio",
        "NOME DO MUNICÍPIO": "municipio",
        "POPULAÇÃO ESTIMADA": "populacao",
    }
)

colunas_esperadas = {
    "uf",
    "cod_uf",
    "cod_municipio",
    "municipio",
    "populacao",
}

colunas_faltantes = colunas_esperadas - set(df_ibge.columns)

if colunas_faltantes:
    raise ValueError(
        f"Colunas esperadas não encontradas: {colunas_faltantes}"
    )

print("Colunas após padronização:")
print(df_ibge.columns.tolist())

Colunas após padronização:
['uf', 'cod_uf', 'cod_municipio', 'municipio', 'populacao']


## 11. Tratamento dos códigos municipais

In [11]:
df_ibge["cod_uf"] = (
    df_ibge["cod_uf"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(2)
)

df_ibge["cod_municipio"] = (
    df_ibge["cod_municipio"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

df_ibge["codigo_ibge_7"] = (
    df_ibge["cod_uf"]
    + df_ibge["cod_municipio"]
)

df_ibge[
    [
        "uf",
        "cod_uf",
        "cod_municipio",
        "codigo_ibge_7",
        "municipio"
    ]
].head()

,uf,cod_uf,cod_municipio,codigo_ibge_7,municipio
0,RO,11,00015,1100015,Alta Floresta D'Oeste
1,RO,11,00023,1100023,Ariquemes
2,RO,11,00031,1100031,Cabixi
3,RO,11,00049,1100049,Cacoal
4,RO,11,00056,1100056,Cerejeiras


## 12. Tratamento da população

In [12]:
df_ibge["populacao"] = (
    df_ibge["populacao"]
    .astype("string")
    .str.replace(r"\(\d+\)\s*$", "", regex=True)
    .str.replace(".", "", regex=False)
    .str.strip()
)

df_ibge["populacao"] = pd.to_numeric(
    df_ibge["populacao"],
    errors="coerce"
).astype("Int64")

df_ibge[["municipio", "populacao"]].head()

,municipio,populacao
0,Alta Floresta D'Oeste,22516
1,Ariquemes,111148
2,Cabixi,5067
3,Cacoal,86416
4,Cerejeiras,16088


## 13. Investigação dos registros incompletos

In [15]:
print("Dimensão antes da limpeza:", df_ibge.shape)

print("\nValores nulos:")
print(df_ibge.isna().sum())

registros_incompletos = df_ibge[
    df_ibge[
        [
            "cod_uf",
            "cod_municipio",
            "municipio",
            "populacao",
            "codigo_ibge_7"
        ]
    ].isna().any(axis=1)
].copy()

print("\nQuantidade de registros incompletos:", len(registros_incompletos))
registros_incompletos

Dimensão antes da limpeza: (5593, 6)

Valores nulos:
uf                2
cod_uf           23
cod_municipio    23
municipio        23
populacao        23
codigo_ibge_7    23
dtype: int64

Quantidade de registros incompletos: 23


,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7
5570,NaN,<NA>,<NA>,NaN,<NA>,<NA>
5571,Fonte: IBGE. Diretoria de Pesquisas - DPE - C...,<NA>,<NA>,NaN,<NA>,<NA>
5572,NaN,<NA>,<NA>,NaN,<NA>,<NA>
5573,Notas:,<NA>,<NA>,NaN,<NA>,<NA>
5574,(1) População judicial do município de Porto V...,<NA>,<NA>,NaN,<NA>,<NA>
5575,(2) População judicial do município de Benjami...,<NA>,<NA>,NaN,<NA>,<NA>
5576,(3) População judicial do município de Caapira...,<NA>,<NA>,NaN,<NA>,<NA>
5577,(4) População judicial do município de Guajará...,<NA>,<NA>,NaN,<NA>,<NA>
5578,(5) População judicial do município de Jutaí-A...,<NA>,<NA>,NaN,<NA>,<NA>
5579,(6) População judicial do município de Lábrea-...,<NA>,<NA>,NaN,<NA>,<NA>


## 14. Identificação dos registros municipais válidos

In [16]:
mascara_municipio_valido = (
    df_ibge["uf"].str.fullmatch(r"[A-Z]{2}", na=False)
    & df_ibge["cod_uf"].str.fullmatch(r"\d{2}", na=False)
    & df_ibge["cod_municipio"].str.fullmatch(r"\d{5}", na=False)
    & df_ibge["codigo_ibge_7"].str.fullmatch(r"\d{7}", na=False)
    & df_ibge["municipio"].notna()
    & df_ibge["populacao"].notna()
)

registros_nao_municipais = df_ibge.loc[
    ~mascara_municipio_valido
].copy()

print(
    "Registros fora do padrão municipal:",
    len(registros_nao_municipais)
)

registros_nao_municipais

Registros fora do padrão municipal: 23


,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7
5570,NaN,<NA>,<NA>,NaN,<NA>,<NA>
5571,Fonte: IBGE. Diretoria de Pesquisas - DPE - C...,<NA>,<NA>,NaN,<NA>,<NA>
5572,NaN,<NA>,<NA>,NaN,<NA>,<NA>
5573,Notas:,<NA>,<NA>,NaN,<NA>,<NA>
5574,(1) População judicial do município de Porto V...,<NA>,<NA>,NaN,<NA>,<NA>
5575,(2) População judicial do município de Benjami...,<NA>,<NA>,NaN,<NA>,<NA>
5576,(3) População judicial do município de Caapira...,<NA>,<NA>,NaN,<NA>,<NA>
5577,(4) População judicial do município de Guajará...,<NA>,<NA>,NaN,<NA>,<NA>
5578,(5) População judicial do município de Jutaí-A...,<NA>,<NA>,NaN,<NA>,<NA>
5579,(6) População judicial do município de Lábrea-...,<NA>,<NA>,NaN,<NA>,<NA>


## 15. Limpeza dos registros não municipais

Os registros identificados na etapa anterior correspondem a linhas auxiliares
do arquivo original como fonte, notas e observações e não representam
municípios.

Esses registros serão removidos para manter no dataset apenas observações
municipais válidas.

In [17]:
df_ibge = (
    df_ibge.loc[mascara_municipio_valido]
    .copy()
    .reset_index(drop=True)
)

print("Dimensão após a limpeza:", df_ibge.shape)

Dimensão após a limpeza: (5570, 6)


## 16. Validação final dos dados

Após a limpeza, o dataset contém apenas registros municipais completos. Não foram identificados valores nulos, códigos IBGE duplicados ou códigos fora do padrão esperado de 7 dígitos.

In [19]:
print("Dimensão final:", df_ibge.shape)

print("\nValores nulos:")
print(df_ibge.isna().sum())

codigos_duplicados = df_ibge[
    df_ibge["codigo_ibge_7"].duplicated(keep=False)
].sort_values("codigo_ibge_7")

print(
    "\nCódigos IBGE duplicados:",
    df_ibge["codigo_ibge_7"].duplicated().sum()
)

print(
    "Códigos IBGE inválidos:",
    (
        ~df_ibge["codigo_ibge_7"]
        .str.fullmatch(r"\d{7}", na=False)
    ).sum()
)

codigos_duplicados

Dimensão final: (5570, 6)

Valores nulos:
uf               0
cod_uf           0
cod_municipio    0
municipio        0
populacao        0
codigo_ibge_7    0
dtype: int64

Códigos IBGE duplicados: 0
Códigos IBGE inválidos: 0


,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7


## 17. Municípios do estado de Goiás

O recorte do estado de Goiás contém 246 registros municipais. Esse subconjunto poderá ser utilizado posteriormente nas análises específicas de internações, estabelecimentos e disponibilidade de leitos.

In [20]:
df_goias = df_ibge.loc[
    df_ibge["uf"] == "GO"
].copy()

print(
    "Quantidade de municípios de Goiás:",
    len(df_goias)
)

df_goias.head()

Quantidade de municípios de Goiás: 246


,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7
5323,GO,52,00050,Abadia de Goiás,9158,5200050
5324,GO,52,00100,Abadiânia,20873,5200100
5325,GO,52,00134,Acreúna,22710,5200134
5326,GO,52,00159,Adelândia,2515,5200159
5327,GO,52,00175,Água Fria de Goiás,5843,5200175


## 18. Preparação para integração com SIH/SUS e CNES

O código municipal do IBGE foi padronizado para sete dígitos e será utilizado
posteriormente para verificar a compatibilidade com os códigos presentes nos
dados do SIH/SUS e CNES.

A validação efetiva da compatibilidade será realizada na etapa de integração
dos datasets.

## 19. Conclusão

A exploração dos dados populacionais do IBGE permitiu identificar e preparar a estrutura necessária para utilização da população municipal nas etapas posteriores do projeto.

Após o tratamento:

- foram mantidos apenas os registros correspondentes a municípios
- os códigos municipais foram padronizados no formato IBGE de 7 dígitos
- a população estimada foi convertida para formato numérico
- não foram identificados valores nulos nos registros municipais válidos
- não foram identificados códigos municipais duplicados
- não foram identificados códigos IBGE fora do padrão esperado
- foi criado um recorte específico dos municípios do estado de Goiás

O dataset tratado poderá ser utilizado posteriormente na integração com os dados do SIH/SUS e CNES para construção de indicadores relativos à população.